# BERTopic Topic Modeling for r/CryptoScams

This notebook builds topics using BERTopic on Reddit posts (title + selftext).

In [7]:
from pathlib import Path
import json
import re

import pandas as pd
import numpy as np
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    nltk_available = True
except Exception:
    nltk_available = False
    stopwords = None
    WordNetLemmatizer = None

print(f"NLTK available: {nltk_available}")

NLTK available: True


In [8]:
data_path = Path("../CryptoScams/filtered_r_CryptoScams_2020-2025_posts.jsonl")
rows = []

with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        rows.append({
            "id": record.get("id", ""),
            "date": record.get("date", ""),
            "title": record.get("title", ""),
            "selftext": record.get("selftext", ""),
        })

df = pd.DataFrame(rows)
df.head()

,id,date,title,selftext
0,q0uafx,"October 04, 2021 00:18:21 UTC",My Story (with COBOS vip),"Hi everyone, I wish I didn't have to write thi..."
1,t77deq,"March 05, 2022 11:15:24 UTC",XHEX investment scam from Tinder date,I was scammed by people who pretented to be a ...
2,w37rbh,"July 20, 2022 00:10:06 UTC",Possible Romance Scam?,Hello since i almost got scammed from a previo...
3,wzsn9j,"August 28, 2022 10:54:12 UTC",bit-times.net fake trading site,This site is part of a Romance Pig Butchering ...
4,x2tbuo,"September 01, 2022 00:29:11 UTC",New scam website,I met a romance scam. I was directed to this w...


In [9]:
df["title"] = df["title"].fillna("")
df["selftext"] = df["selftext"].fillna("")
df["text"] = (df["title"] + " " + df["selftext"]).str.strip()
df = df[df["text"] != ""].copy()

sample_size = 400
df = df.head(sample_size).reset_index(drop=True)
df[["id", "date", "text"]].head()

,id,date,text
0,q0uafx,"October 04, 2021 00:18:21 UTC","My Story (with COBOS vip) Hi everyone, I wish ..."
1,t77deq,"March 05, 2022 11:15:24 UTC",XHEX investment scam from Tinder date I was sc...
2,w37rbh,"July 20, 2022 00:10:06 UTC",Possible Romance Scam? Hello since i almost go...
3,wzsn9j,"August 28, 2022 10:54:12 UTC",bit-times.net fake trading site This site is p...
4,x2tbuo,"September 01, 2022 00:29:11 UTC",New scam website I met a romance scam. I was d...


In [10]:
url_re = re.compile(r"https?://\S+|www\.\S+")
non_letters_re = re.compile(r"[^a-zA-Z\s]")

stop_words = set("""
 a about above after again against all am an and any are aren't as at be
 because been before being below between both but by can can't cannot could couldn't did didn't do does doesn't
 doing don't down during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's
 her here here's hers herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's
 me more most mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't
 she she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
 these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're we've
 were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't you you'd
 you'll you're you've your yours yourself yourselves
""".split())

if nltk_available:
    try:
        stop_words.update(stopwords.words("english"))
        lemmatizer = WordNetLemmatizer()
    except LookupError:
        nltk_available = False
        lemmatizer = None
else:
    lemmatizer = None

def preprocess(text):
    text = text.lower()
    text = url_re.sub(" ", text)
    text = non_letters_re.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return ""
    tokens = text.split()
    if nltk_available and lemmatizer is not None:
        tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    else:
        tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(preprocess)
df = df[df["clean_text"] != ""].reset_index(drop=True)
df[["id", "clean_text"]].head()

,id,clean_text
0,q0uafx,story cobos vip hi everyone wish didn write po...
1,t77deq,xhex investment scam tinder date scammed peopl...
2,w37rbh,possible romance scam hello since almost got s...
3,wzsn9j,bit times net fake trading site site part roma...
4,x2tbuo,new scam website met romance scam directed web...


In [11]:
embedding_model = "all-MiniLM-L6-v2"
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=10, metric="euclidean", prediction_data=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=10,
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(df["clean_text"].tolist())
topics[:10]

2026-05-28 11:20:40,330 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2026-05-28 11:20:41,693 - BERTopic - Embedding - Completed ✓
2026-05-28 11:20:41,693 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-28 11:20:42,567 - BERTopic - Dimensionality - Completed ✓
2026-05-28 11:20:42,569 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-28 11:20:42,587 - BERTopic - Cluster - Completed ✓
2026-05-28 11:20:42,590 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-28 11:20:42,632 - BERTopic - Representation - Completed ✓


[1, 0, 0, -1, 0, 0, 2, 0, -1, 0]

In [12]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,130,-1_scam_butchering_pig_money,"[scam, butchering, pig, money, crypto, get, re...",[convince mother law scammed whatsapp group mo...
1,0,150,0_money_scam_crypto_will,"[money, scam, crypto, will, just, funds, accou...",[xhex investment scam tinder date scammed peop...
2,1,37,1_whatsapp_group_scam_pig,"[whatsapp, group, scam, pig, butchering, crypt...",[minimize potential harm caused pig butchering...
3,2,22,2_scam_pig_butchering_money,"[scam, pig, butchering, money, funds, scammed,...",[withdrawal scam website deposit scam recently...
4,3,22,3_pig_butchering_scam_beware,"[pig, butchering, scam, beware, scams, man, ge...",[make money pig butchering scam hi everyone re...
5,4,16,4_butchering_pig_scam_friend,"[butchering, pig, scam, friend, crypto, back, ...",[fell pig butchering scam receive btc back ok ...


In [16]:
outputs_dir = Path("../outputs")
outputs_dir.mkdir(exist_ok=True)

topic_info.to_csv(outputs_dir / "topics.csv", index=False)

doc_topics = df[["id", "date"]].copy()
doc_topics["topic"] = topics

if probs is None:
    doc_topics["probability"] = None
else:
    doc_topics["probability"] = [
        float(p.max()) if topic != -1 else None
        for topic, p in zip(topics, probs)
    ]

doc_topics["title"] = (
    df["title"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
 )
doc_topics["selftext"] = (
    df["selftext"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
 )
doc_topics.to_csv(outputs_dir / "doc_topics.csv", index=False)
doc_topics.head()

,id,date,topic,probability,title,selftext
0,q0uafx,"October 04, 2021 00:18:21 UTC",1,0.390462,My Story (with COBOS vip),"Hi everyone, I wish I didn't have to write thi..."
1,t77deq,"March 05, 2022 11:15:24 UTC",0,0.585219,XHEX investment scam from Tinder date,I was scammed by people who pretented to be a ...
2,w37rbh,"July 20, 2022 00:10:06 UTC",0,0.641104,Possible Romance Scam?,Hello since i almost got scammed from a previo...
3,wzsn9j,"August 28, 2022 10:54:12 UTC",-1,NaN,bit-times.net fake trading site,This site is part of a Romance Pig Butchering ...
4,x2tbuo,"September 01, 2022 00:29:11 UTC",0,1.000000,New scam website,I met a romance scam. I was directed to this w...


## Adjustments

- Sample size: change `sample_size` in the sampling cell.
- Topic size: change `min_topic_size` in the BERTopic setup.
- Embeddings: change `embedding_model` (for example, `all-mpnet-base-v2`).